# Sales Batch-Only Pipeline

**Bronze (3):** `sales_bz`, `products_bz`, `customers_bz`
**Silver (4):** `sales`, `products`, `customers`, `sales_enriched`
**Gold (3):** `sales_daily_summary`, `customer_sales_summary`, `product_performance` (view)

- No streaming. Ingestion uses `COPY INTO`; transforms use MERGE with `SET *` / `INSERT *`.
- `products` and `customers` arrive as **CDC feeds** (`operation` + `seqNum` / `sequenceNum`)
  and are maintained as **SCD Type 1** dimensions: latest sequence wins, deletes are applied.
- `sales` is an **immutable fact** — insert-only MERGE, duplicates ignored.
- Each table is isolated in its own cell to **run and display separately**.

## 0) Environment / Locations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lit, expr, upper, rank, to_date, to_timestamp
from pyspark.sql.utils import AnalysisException

catalog = "abs_dev"
db_name = "abs_db"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{db_name}")
spark.sql(f"USE {catalog}.{db_name}")

# External Locations
base_dir_data = spark.sql("DESCRIBE EXTERNAL LOCATION `abs-external-location`").select("url").collect()[0][0]
try:
    base_dir_checkpoint = spark.sql("DESCRIBE EXTERNAL LOCATION `landing-zone`").select("url").collect()[0][0]
except AnalysisException:
    base_dir_checkpoint = None
test_data_dir = base_dir_data + "/test_data"

landing_zone = base_dir_data + "/raw"
print("Landing:", landing_zone)

Landing: abfss://abs-container@silastorage.dfs.core.windows.net//raw


### Key typing note
`order_id`, `customer_id`, `product_id` and `transaction_id` are kept as **STRING** end to end.
This survives non-numeric keys (`CUST-001`, `P_1042`) which would otherwise silently cast to NULL
and break every join downstream. If your keys are guaranteed numeric, change the silver DDL to
`BIGINT` and wrap the source selects in `try_cast(... AS BIGINT)`.

## 1) Reference Table — date_lookup

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.date_lookup (
  date date,
  week int,
  year int,
  month int,
  dayofweek int,
  dayofmonth int,
  dayofyear int,
  week_part string
) USING DELTA
""")

print("creating date_lookup table...", end='')

spark.sql(f"""INSERT OVERWRITE TABLE {catalog}.{db_name}.date_lookup
SELECT
  d                                                                         AS date,
  weekofyear(d)                                                             AS week,
  year(d)                                                                   AS year,
  month(d)                                                                  AS month,
  dayofweek(d)                                                              AS dayofweek,
  dayofmonth(d)                                                             AS dayofmonth,
  dayofyear(d)                                                              AS dayofyear,
  CASE WHEN dayofweek(d) IN (1, 7) THEN 'weekend' ELSE 'weekday' END        AS week_part
FROM (SELECT explode(sequence(DATE '2018-01-01', DATE '2030-12-31', INTERVAL 1 DAY)) AS d)
""")
print("Done")

display(spark.read.table(f"{catalog}.{db_name}.date_lookup").limit(20))

creating date_lookup table...Done


date,week,year,month,dayofweek,dayofmonth,dayofyear,week_part
2018-01-01,1,2018,1,2,1,1,weekday
2018-01-02,1,2018,1,3,2,2,weekday
2018-01-03,1,2018,1,4,3,3,weekday
2018-01-04,1,2018,1,5,4,4,weekday
2018-01-05,1,2018,1,6,5,5,weekday
2018-01-06,1,2018,1,7,6,6,weekend
2018-01-07,1,2018,1,1,7,7,weekend
2018-01-08,2,2018,1,2,8,8,weekday
2018-01-09,2,2018,1,3,9,9,weekday
2018-01-10,2,2018,1,4,10,10,weekday
